# Orbit Wars — RL Modeli ve Eğitim Notları

Bu notebook, repoda güncel olan **PlanetOnly Set Transformer + PPO + delayed reward** yığınının teorik temellerini ve pratik ayrıntılarını toplu halde açıklar. Odak, kod gezisi değil; **neden böyle tasarlandığının matematiksel ve RL teorisindeki karşılığıdır.**

İlgili dosyalar:

- `model.py` — full Set Transformer policy/value ağı.
- `model_planet_only.py` — fleet token kullanmayan sadeleştirilmiş varyant (şu an default).
- `obs_builder.py` — gözlem vektörü ve edge bias üretimi.
- `rl/ppo.py` — clipped surrogate PPO güncellemesi.
- `rl/buffer.py` — on-policy rollout buffer, GAE, `add_delayed_reward`.
- `rl/rewards.py` — reward konfigürasyonu ve step shaping.
- `rl/rollout.py` — self-play episode toplayıcı, delayed reward dağıtımı.
- `rl/production_reward_tracker.py`, `rl/fleet_tracker.py`, `rl/defense_tracker.py`, `rl/planet_loss_tracker.py` — delayed credit assignment mekanizmaları.

Notebook çalıştırılabilir değildir; amacı kuramsal bir referans olmaktır. Bazı kısa kod blokları, kod tabanına bir-bire-bir bağlantı kurmak için eklenmiştir.

## 1. Problem: Self-Play Zero-Sum MDP

Orbit Wars, iki (veya 4) oyunculu, ayrık zamanlı, kısmen gözlemlenebilir ama her oyuncuya simetrik bilgi veren bir strateji oyunudur. Eğitim açısından onu aşağıdaki gibi formalize ediyoruz:

- Durum: $s_t$ — environment’ın tam durumu (gezegenler, filolar, üretim, komet vs).
- Oyuncu $i$ için gözlem: $o_t^{(i)} = \phi_i(s_t)$ — kendi perspektifine göre normalize edilmiş feature tensörleri (`obs_builder.build_observation`).
- Oyuncu $i$ için aksiyon: $a_t^{(i)} = (\text{src}, \text{tgt}, \text{ship\_bin}, \text{is\_noop})$.
- Geçiş: $s_{t+1} = T(s_t, a_t^{(0)}, a_t^{(1)}, \dots)$ — tüm oyuncu aksiyonları eşzamanlı uygulanır.
- Ödül: $r_t^{(i)}$ — terminal galibiyet/mağlubiyet + dense shaping + delayed credit.

**Self-play**: tek bir politika $\pi_\theta$ her iki tarafta da oynatılır. Bu, **non-stationary ama simetrik** bir ortama sebep olur; çünkü $\pi$ güncellendikçe rakibin davranışı da değişir. PPO’nun on-policy özelliği ve küçük güncelleme adımı (clip), bu kaymayı tolere etmek için tasarımın tutarlı kısmıdır.

Amaç: Oyuncunun kendi ödül toplamının beklentisini maksimize etmek.

$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T} \gamma^{\,t}\, r_t \right].
$$

Zero-sum yapısı nedeniyle bir oyuncunun değeri, diğerinin değerinin yaklaşık olarak negatifidir; terminal ödülde $+1/-1$ simetrisi ile bu açıkça korunur (`normalize_terminal_rewards`).

## 2. Gözlem (Observation) Tasarımı

`obs_builder.py` env durumunu sabit şekilli numpy tensörlerine çevirir. Bu tensörler Set Transformer’in girdisidir. Temel ilkeler:

1. **Permutation invariance**: gezegen ve filo slotları set olarak sunulur, maske ile padding ayırt edilir.
2. **Kutupsal simetri**: oyunun merkezinde güneş var; özellikler $(r,\theta)$ tabanlı çıkarılır, $(\cos\theta, \sin\theta)$ ile döngüsellik korunur.
3. **Geometrik ön bilgi**: modelin uzaklık/açı öğrenmesi için **edge bias** tensörü hazırlanır ve attention’a eklenir.
4. **Aksiyon maskeleme**: yalnızca yeterli gemisi olan kendi gezegenlerimiz kaynak olabilir (`source_mask`).

### 2.1 Planet features (`PLANET_FEATURE_DIM = 31`)

Her gezegen için aşağıdaki özellikler üretilir (indeksler `obs_builder` içindeki `PF_IDX_*` sabitleriyle eşleşir):

- Sahiplik: `is_self`, `is_enemy`, `is_neutral` (one-hot).
- Ekonomi: normalize edilmiş gemi sayısı, üretim, yarıçap, `ships_to_cap`, `log_ships`.
- Konum: $r/50$, $\cos\theta$, $\sin\theta$, türler (static / rotating / comet), $\omega$.
- **Incoming fleet özeti (planet-only modelde kritik)**: kendi ve rakip filolarının ETA bucket’ları (`[0,5], (5,15], (15,30], 30+`), toplam gelen kuvvet, minimum ETA, en büyük düşman filosunun büyüklük/ETA’sı ve `net_threat = enemy_total - ships - self_total` gibi aggregate tehdit sinyalleri.

Bu zenginleştirme sayesinde planet-only model, fleet token’ları olmadan da savunma/saldırı zamanlamasını kavrayabilir. Eski 23-boyutlu checkpoint’ler `load_checkpoint` içinde state_dict’ten dinamik olarak tespit edilir (`planet_proj.weight.shape[1]`).

### 2.2 Fleet features

`f_max` slotluk filo tensörü üretilir (`FLEET_FEATURE_DIM = 11`): sahiplik, gemi sayısı, kaynak/hedef kodları, kalan ETA, kutupsal konum vb. PlanetOnly variant’ta `f_max = 0` olduğu için model bu tensörü kullanmaz; `full` variant’ta her filo ayrı token olarak kodlanır.

### 2.3 Global features

`GLOBAL_FEATURE_DIM = 12` boyutlu vektör: oyun turu, sahip olduğum/rakibin gezegen sayısı, toplam gemi oranı, pozisyonsal özet ve harcanmış süre gibi CLS-token benzeri global özetler içerir. Bu tek token, transformer’ın en sonuna eklenir ve `value head`’in girdisidir.

### 2.4 Edge bias

Her iki token arasında 5 boyutlu bir edge özelliği hesaplanır:

$$
e_{ij} = (d_{ij}/d_{\max},\; \Delta x,\; \Delta y,\; \sin\Delta\varphi_{\text{sun}},\; \cos\Delta\varphi_{\text{sun}}).
$$

Aynı yörünge hızına sahip iki gezegen için $\Delta\varphi_{\text{sun}}$ **zaman-değişmez** bir büyüklüktür; bu da modelin yörünge ilişkilerini öğrenmesini kolaylaştırır. Edge bias, **head başına additive attention bias** olarak projekte edilir:

$$
\text{scores} = \frac{QK^\top}{\sqrt{d}} + B_{\text{edge}}\;, \quad B_{\text{edge}} \in \mathbb{R}^{H\times T\times T}.
$$

Bu, modelin koordinatlardan mesafe yeniden öğrenmesini beklemek yerine geometriyi doğrudan önsel bilgi olarak vermek demek.

## 3. Model: Set Transformer Policy / Value

`OrbitWarsPolicy` (full) ve `PlanetOnlyOrbitWarsPolicy` aynı çekirdeği paylaşır. Token düzeni:

- **Full**: `[planet_0, ..., planet_{P-1}, fleet_0, ..., fleet_{F-1}, GLOBAL]`, yani $T = P_{\max} + F_{\max} + 1$.
- **PlanetOnly**: `[planet_0, ..., planet_{P-1}, GLOBAL]`, yani $T = P_{\max} + 1$.

Her tokena tip embedding eklenir (planet/fleet/global). Bu, heterojen tokenları $d_\text{model}$ uzayında ayırt etmeyi kolaylaştırır.

### 3.1 Biased multi-head attention

Standart çok kafalı dikkat mekanizmasına ek olarak, her pozisyon çifti için additive bias:

$$
\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + B_{\text{edge}} + M_{\text{pad}}\right) V,
$$

- $B_{\text{edge}}$: yukarıdaki geometrik bias.
- $M_{\text{pad}}$: padded key’ler için $-\infty$ maskesi.

Padded satırlarda softmax NaN üretir; `torch.nan_to_num` ile güvenli sıfırla doldurulur.

### 3.2 Pre-norm Transformer blokları

$$
x \leftarrow x + \text{Attn}(\text{LN}(x))\;,\quad x \leftarrow x + \text{FFN}(\text{LN}(x)).
$$

Varsayılan: `d_model=128`, `n_heads=4`, `n_layers=4`, FFN genişleme çarpanı 4 ve GELU aktivasyon. Bu ölçek, Orbit Wars gibi küçük küme uzayları için yeterince ifadeli, aynı zamanda CPU’da smoke test yapılabilecek kadar küçüktür.

### 3.3 Policy head’leri ve aksiyon faktörizasyonu

Aksiyon üç bağımsız fakat **koşullu** kategorik dağılımdan oluşur:

$$
\pi(a \mid s) = p_{\text{src}}(\text{src}\mid s)\; p_{\text{tgt}}(\text{tgt}\mid s, \text{src})\; p_{\text{ships}}(\text{bin}\mid s, \text{src}, \text{tgt}).
$$

Bu faktörizasyon **action space patlamasını** ($P_{\max}^2 \times n_\text{bins}$) her bir faktör için doğrusal hale indirir ve yasal olmayan kombinasyonların (kendi gezegeni hedefleme, boş gezegenden launch, vb.) doğrudan maskelenmesine izin verir.

- **`src_head`**: her planet token’ı için skor + öğrenilebilir **NO-OP** logit. Maskesiz kaynaklar $-\infty$’a basılır.
- **`tgt_head`** (pointer attention): kaynak token $q_s$ ile her hedef token $k_t$ arasında
  $\text{score}(s,t) = q_s^\top k_t / \sqrt{d}$. Kendine hedef verme ve padded slotlar $-\infty$ ile maskelenir.
- **`ship_head`**: kaynak token’ına bağlı 5 ayrık bin (örn. ships *0.2, 0.4, 0.6, 0.8, 1.0 gibi).
- **`value_head`**: global token’ına MLP; $V^\pi(s)$ tahmini.

Log-olasılık ve entropi NO-OP durumu için **geçerli sadece `src` faktörüdür**: target ve ship bin’in log-prob/entropi katkısı `valid = ~is_noop` maskesiyle silinir. Bu, `evaluate_source_actions` ve `act_source` içinde tutarlı şekilde uygulanır, dolayısıyla PPO güncellemesi NO-OP için değerlendirme tutarlılığını korur.

### 3.4 Neden iki variant?

- **Full**: filo trajektorileri token olarak verilir. Bu, interception ve timing gerektiren ileri düzey stratejiler için ifade gücü sağlar; ama token sayısı arttığı için daha yavaş.
- **PlanetOnly (şu anki default)**: gezegen özelliklerinin içine ETA-bucket’lı incoming fleet özeti gömülür. Model daha küçük ve hızlıdır. Çoğu Orbit Wars stratejisi için yeterli bulunmuştur; `--model-variant full` ile full moda geçmek hâlâ mümkündür.

Model seçimi `train.py` içindeki `_make_policy` ve `model.py`/`model_planet_only.py` dosyalarında; `obs_builder.build_observation(..., planet_feature_dim=policy.planet_dim)` ile dinamik olarak eşleştirilir (bkz. `rl/rollout.py`, `rl/watch.py`).

In [ ]:
# Mimari hızlı özet (çalıştırmak için opsiyonel)
from model_planet_only import PlanetOnlyOrbitWarsPolicy
from obs_builder import PLANET_FEATURE_DIM, P_MAX_DEFAULT

policy = PlanetOnlyOrbitWarsPolicy(d_model=128, n_heads=4, n_layers=4)
total_params = sum(p.numel() for p in policy.parameters())
print(f"Variant: {policy.model_variant}")
print(f"Token count T = {policy.p_max + 1}, planet_dim = {policy.planet_dim}")
print(f"Parametre sayısı: {total_params:,}")


## 4. Öğrenme Algoritması: PPO + GAE

Politika güncellemesi **Proximal Policy Optimization (clip variant)** ile yapılır (`rl/ppo.py`).

### 4.1 Clipped surrogate objective

$r_t(\theta) = \pi_\theta(a_t|s_t) / \pi_{\theta_{\text{old}}}(a_t|s_t)$ olmak üzere:

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[\min\!\left(r_t\hat A_t,\; \text{clip}(r_t, 1-\epsilon, 1+\epsilon) \hat A_t\right)\right].
$$

Bu terim, politika dağılımının **bir adımda çok fazla kaymasını** yapısal olarak cezalandırır: $r_t$ $[1-\epsilon, 1+\epsilon]$ aralığını aşarsa clip devreye girer ve gradient o örnekten gelmez.

### 4.2 Value loss (opsiyonel clipping)

$$
L^{VF}(\theta) = \frac{1}{2}\max\left((V_\theta - R)^2,\; (V^{\text{clip}} - R)^2\right),
$$

burada $V^{\text{clip}} = V_{\theta_{\text{old}}} + \text{clip}(V_\theta - V_{\theta_{\text{old}}}, -\epsilon_V, \epsilon_V)$. Value clipping, critic’in ani büyük güncellemelerini engeller.

### 4.3 Entropi bonusu

Toplam kayıp:

$$
L(\theta) = L^{\text{CLIP}} + c_1 L^{VF} - c_2 H[\pi_\theta](s).
$$

Default: $c_1 = 0.5$, $c_2 = 0.01$. Entropy terimi, erken aşamada exploration’ı korur.

### 4.4 Generalized Advantage Estimation

GAE-$\lambda$ yüksek varyanslı Monte Carlo getirisi ile düşük varyanslı TD(0) arasında sürekli bir interpolasyon sağlar:

$$
\delta_t = r_t + \gamma V(s_{t+1})\,\mathbb{1}[\neg\text{done}_t] - V(s_t),\qquad
\hat A_t = \sum_{l \geq 0} (\gamma\lambda)^l \delta_{t+l}.
$$

`rl/buffer.py` içinde her oyuncunun transition’ları için ayrı rekürsif hesaplama yapılır (`compute_gae`). Default: `gamma=0.99`, `lam=0.95`.

### 4.5 Advantage normalizasyonu

Her mini-batch öncesi advantage’lar ortalama=0, std=1’e normalize edilir. Bu, PPO clip aralığının optimizer boyunca tutarlı davranmasını sağlar.

## 5. Ödül (Reward) Sistemi

Ödül, üç bileşenden oluşur ve `rl/rewards.py` + `rl/rollout.py` içinde yönetilir.

### 5.1 Terminal ödül

Env sonu ödülü $\pm 1$ (kazanan/kaybeden), berabere için normalize edilir (`normalize_terminal_rewards`). Bu, zero-sum özelliğini koruyan temel sinyaldir.

### 5.2 Dense step shaping

`compute_step_reward` her transition için opsiyonel olarak:

- **Capture-value**: planetin üretim değeri ve kalan turan oranında anlık kazanım/kayıp.
- **Ship-diff / ship-loss**: gemi avantajı ve kayıp cezası.
- **Step cost / invalid action**: çok ince disiplin terimleri.

`production_simple` presetinde step shaping çoğunlukla kapatılır; kredi ana olarak **delayed trackers** ile verilir.

### 5.3 Delayed credit (trackers)

Fleet launch ve NO-OP gibi **etkisi gelecekte görülen** aksiyonlar için doğrudan step ödülü kredili değildir: event gerçekleşince ödül geriye, onu tetikleyen transition’a eklenir.

- `rl/fleet_tracker.py` — varış olayı: capture / enemy kill / wasted / self reinforce.
- `rl/production_reward_tracker.py` — gezegen ele geçirme ve planet loss.
- `rl/defense_tracker.py` — NO-OP ile tehdidi absorbe eden hold ödülü.
- `rl/planet_loss_tracker.py` — kaynaktan gemi çekildikten sonra o gezegenin düşmesi cezası.

Tüm bu trackerlar çıktıyı **event turn** ile birlikte `RolloutBuffer.add_delayed_reward(idx, reward, event_turn=t, discount=...)` olarak buffer’a yazar.

### 5.4 Delayed reward discount

Bir aksiyonun ödülü $\Delta t = t_{\text{event}} - t_{\text{action}}$ step sonra geliyorsa, buffer bunu

$$
r_{\text{credited}} = r \cdot d^{\Delta t}
$$

olarak ekler (`add_delayed_reward`, bkz. `rl/buffer.py`). Burada $d = $ `delayed_reward_discount`.

- $d = 1$ (şu anki default, hem `current` hem `production_simple`): hiç decay yok; ödül tam değerinde launch transition’ına eklenir. Co-launch olaylarında ödül **eşit ship-share** ile dağıtılır.
- $d < 1$: gecikme arttıkça etkin kredi azalır; eski launch’lar daha az pay alır.

`production_simple`’ın $d = 1$’e kalibre edildiğini unutma. Discount’u 1’in altına almak istiyorsan, capture/loss formülünün **zaten** zaman faktörü içerdiğini hatırla (bkz. 5.5); iki kez decay uygulamak istemeyebilirsin.

### 5.5 Capture ve loss ödülleri (production_simple)

`compute_production_capture_reward` ve `compute_production_planet_loss_penalty` **normalize** formülü kullanır:

$$
R_{\text{capture}}(s, t) = p(s)\cdot \frac{T_{\max} - t}{T_{\max}}\cdot m_{\text{opp}}^{\mathbb{1}[\text{enemy}]}\cdot \frac{n_i}{\sum_j n_j}.
$$

Burada:

- $p(s) \in [1, 5]$: gezegenin üretim değeri.
- $(T_{\max}-t)/T_{\max} \in [0, 1]$: kalan oyun yüzdesi (zaman avantajı sinyali).
- $m_{\text{opp}} = (n_{\text{opp}}+1)/n_{\text{opp}}$: enemy capture multiplier (2-player için 2.0, 4-player 4-opp için 1.25).
- $n_i / \sum_j n_j$: ship-share (co-launch durumunda).

Tipik değerler ($p = 3$, $t = 100$, $T_{\max} = 500$, alone, 2-player):

- Neutral capture: $3 \cdot 0.8 \cdot 1 \cdot 1 = 2.4$.
- Enemy capture: $3 \cdot 0.8 \cdot 2.0 \cdot 1 = 4.8$.
- Planet loss (savunma çekildi): $-3 \cdot 0.8 \cdot 2.0 \cdot 1 = -4.8$.

Bu ölçek bilinçli olarak **O(1) seviyesinde** tutulur; aksi halde value head’in tahmin etmesi gereken returns 10⁴ mertebelerine çıkar ve `valL` policy loss’u domine eder (bkz. §7.2).

### 5.6 Combat ödülü (production_simple)

`compute_production_simple_fleet_combat_reward` filo varışında uygulanır:

$$
r_{\text{combat}} = \frac{-\text{own\_waste} + \mathbb{1}[\text{kill ya da enemy capture}]\cdot \text{enemy\_kills}}{100}.
$$

100’e bölünmesi de aynı amaca hizmet eder: ship sayıları tipik olarak 10–50 aralığında olduğu için bölmeden ham değerleri capture ödülünden 10× büyük olurdu. Normalize edilmiş combat ödülü ±0.1–0.5 aralığında kalır.

### 5.7 Airborne maliyeti

Launch anında, **hemen** (delay’sız) eklenen küçük bir maliyet:

$$
r_{\text{airborne}} = -w_{\text{air}} \cdot n_{\text{ships}} \cdot \mathrm{ETA}.
$$

`production_simple` için $w_{\text{air}} = 0.0005$ (yeni). Tipik 20-gemi 50-tur ETA launch için $-0.5$. Bu, capture ödülünün (~2.4) anlamlı bir kesri ama caydırıcı değil; uzun-rota / büyük-fleet spam’ini hafifçe cezalandırır.

## 6. Rollout ve Training Pipeline

Bir epoch (iteration) içinde şu sıra izlenir (`rl/train.py` + `rl/rollout.py`):

1. **Self-play rollout**: tek bir policy her iki tarafta oynatılır.
   - Her oyuncu için `build_observation` ile planet/edge tensörleri üretilir.
   - `policy.act_source` ile aksiyon örneklenir; log-prob ve value saklanır.
   - Transition buffer’a eklenirken `turn=t` de kaydedilir; böylece gelecekteki delayed ödüller doğru step’e bağlanır.
   - Fleet launch’ları ilgili trackerlara **buffer_idx** ile kayıt edilir.
   - Env.step’ten sonra: fleet arrival, defense hold, production capture/loss, planet loss olaylarından gelen ödüller `add_delayed_reward` ile buffer’a yazılır.
2. **GAE**: episode sonu bootstrap $V=0$ ile `compute_gae` çalıştırılır; her oyuncu ayrı ayrı rekürsif olarak işlenir.
3. **PPO update**: $K$ epoch boyunca shuffled minibatchler üzerinde clipped surrogate loss.
4. **Logging/checkpoint**: KL, clip fraction, value loss, entropi, grad norms ve update ratio metrikleri JSONL’e yazılır, `latest.pt` ve `policy_iter*.pt` kaydedilir.

### 6.1 İki-oyunculu buffer

Her env step’inde iki transition eklenir (p0 ve p1). GAE hesaplanırken `compute_gae` oyuncuları ayırıp her biri için ayrı rekürsyon yapar; böylece iki oyuncunun getirileri birbirine karışmaz.

### 6.2 Seed yönetimi

`seed = train_cfg.seed * 100000 + it * 1000 + ep` formülü, iterasyon ve episode ikilileri için deterministik ama çeşitli seedler üretir; aynı başlangıç konfigürasyonuyla eğitim tekrar üretilebilir.

## 7. Hiperparametreler ve Sezgileri

Varsayılan `TrainConfig` ve `PPOConfig` (see `rl/train.py`, `rl/ppo.py`):

| Hiperparametre | Default | Sezgi |
|---|---|---|
| `gamma` | 0.99 | Uzun-vadeli plan: ~100 adımlık horizon. |
| `gae_lambda` | 0.95 | Yüksek MC / düşük TD arasında tipik PPO dengesi. |
| `learning_rate` | 3e-4 | Adam ile Transformer için güvenli başlangıç. |
| `clip_range` | 0.2 | Politika güncelleme büyüklüğü sınırı. |
| `value_clip_range` | 0.2 | Critic aşırı güncellemesini engeller. |
| `ppo_epochs` | 4 | Aynı buffer üzerinden 4 tur geçiş. |
| `batch_size` | 256 | Minibatch boyutu. |
| `entropy_coef` | 0.05 | Exploration baskısı; normalize ödüllerle erken NO-OP collapse'i geciktirmek için 0.01 yerine 0.05 default. |
| `max_grad_norm` | 0.5 | Gradient clipping; stability. |
| `episodes_per_iter` | 1 | Her oyundan sonra update; hızlı iterasyon. |
| `max_episode_steps` | 500 | Env’in doğal episode sınırı. |
| `delayed_reward_discount` | 1.0 (hem `current` hem `production_simple`) | Delayed credit zayıflatma; <1 yapmak için bilinçli sebep gerekir. |
| `fleet_airborne_weight` (production_simple) | 0.0005 | Launch başına küçük caydırıcı. |

### 7.1 KL ve clip fraction teşhisi

- `approx_kl` çok büyük (>0.02–0.04) olursa: learning rate düşür veya `clip_range` küçült. Policy-rollout mesafesi sağlıksız büyümüş demektir.
- `clip_fraction` >%50: PPO çoğu örneği kesiyor, faydalı gradient kalmıyor. Aynı çözümler geçerli.
- `entropy` hızla 0’a inerse (deterministic collapse): `entropy_coef`’i artır (0.05 → 0.10), veya reward shaping gereksiz miktarda bir davranışı sabitliyor olabilir.
- `value_loss` sürekli artıyorsa veya çok büyük başlıyorsa: ödül ölçeği fırlamış olabilir (bkz. §7.2).

### 7.2 Reward ölçeği ve value-loss collapse

PPO toplam kaybı:

$$
L = L^{\text{CLIP}} + c_1\, L^{VF} - c_2\, H[\pi_\theta], \qquad c_1 = 0.5,\; c_2 = 0.05.
$$

Eğer reward ölçeği büyükse (örn. capture ödülü O(1000)), $L^{VF}$ baştan O(10⁵) olur ve $L^{\text{CLIP}}$ tipik olarak $\lesssim 0.1$. Toplam kayıpta value tarafı 10⁶ kat domine eder. Sonuç:

1. Adam tüm gradient bütçesini critic’i fit etmeye harcar.
2. Encoder shared olduğu için **policy head’lere giden encoder gradient’i tamamen critic’in arzularına göre** şekillenir.
3. Policy heads anlamlı gradient alamaz; src logitleri rastgele yöne sürüklenir.
4. Sigmoid-tabanlı per-planet launch kararı (`act_source`) src logitleri bir kez negatife düşünce **her zaman NO-OP** olur.
5. Politikalar deterministik NO-OP’a çöker, entropy 0’a iner, training donar.

Bu nedenle `production_simple` ödülleri **bilerek O(1) ölçeğine normalize** edilmiştir:

- Capture/loss: $p \cdot (T_{\max}-t)/T_{\max} \cdot m_{\text{opp}}\cdot \text{share}$ → ±0–5 aralığı.
- Combat waste/kill: ships / 100 → ±0.1–0.5.
- Airborne: 0.0005 ölçeğinde ~$-0.5$.

Beklenen returns ~10–50, $L^{VF}$ ~0.1–10 olacak. Policy ve value gradient’ları karşılaştırılabilir kalır; actor-critic döngüsü düzgün çalışır.

### 7.3 İkincil collapse: exploration yetersizliği

Reward ölçeği düzeltilse bile, erken eğitimde rasgele launch'ların başarı oranı düşük olduğu için **launch transition'larının ortalama advantage'ı negatif** olabilir. Tipik production_simple sayıları:

- Başarı %15: $0.15 \cdot 3.6 - 0.85 \cdot 0.7 \approx -0.06$ (marjinal negatif).
- Bazen $-4.8$ planet loss penalty (kaynak gezegen düşerse).

Bu durumda policy *“launch yapma daha güvenli”* sonucuna varır, entropy hızla düşer, capture'un değerli olduğunu öğrenmeden NO-OP'a yapışır.

`entropy_coef = 0.05` (eski 0.01 yerine) bu collapse'i geciktirir: entropy bonusunun toplam loss'a katkısı ~5x büyür, exploration daha uzun korunur. Logda `H` 4.0'tan 1.0'a düşene kadar geçen epoch sayısı önemli ölçüde artar.

**Pratik teşhis kuralı**: ilk 20 epoch'ta `H > 2.0` ve `launch/p > 100` korunmalı. Aksi takdirde `entropy_coef`'i daha da artırmayı veya `planet_loss_weight`'i azaltmayı düşün.

## 8. Çalıştırma: Komutlar

### 8.1 production_simple (önerilen)

```bash
orbit_wars_venv/bin/python -m rl.train \
  --reward-system production_simple \
  --model-variant planet_only \
  --device cuda \
  --iterations 1000 \
  --episodes-per-iter 1 \
  --checkpoint-dir checkpoints/production_simple \
  --log-path logs/production_simple.jsonl
```

Bu komut, normalize edilmiş `production_simple` preset’ini kullanır. `--delayed-reward-discount` flag’ine ihtiyaç yoktur; default `1.0`.

### 8.2 CPU smoke test

```bash
orbit_wars_venv/bin/python -m rl.train \
  --reward-system production_simple \
  --model-variant planet_only \
  --device cpu \
  --iterations 10 \
  --episodes-per-iter 1 \
  --checkpoint-dir checkpoints/smoke \
  --log-path logs/smoke.jsonl
```

İlk birkaç epoch’ta `valL` ~10–500 ve `H` ~1.5–3 görmen beklenir. Daha büyük `valL` (≥1000) veya `H ≈ 0` reward kalibrasyonunda sorun olduğunu gösterir.

### 8.3 Resume

```bash
orbit_wars_venv/bin/python -m rl.train \
  --resume-from checkpoints/production_simple/latest.pt \
  --checkpoint-dir checkpoints/production_simple \
  --log-path logs/production_simple.jsonl \
  --reward-system production_simple --model-variant planet_only \
  --device cuda --iterations 2000 --episodes-per-iter 1
```

### 8.4 Replay izleme (watch)

```bash
orbit_wars_venv/bin/python -m rl.watch \
  --checkpoint checkpoints/production_simple/latest.pt \
  --out replays/production_simple.html
```

Bu, `rl/watch.py` üzerinden HTML replay üretir ve tarayıcıda açılır (artifact gitignore edilmiş). Stokastik ve deterministik moda kendi argüman bayrakları vardır.

## 9. Test Matrisi ve Sanity Checks

`tests/test_rl.py` başlıca şunları kanıtlar:

- `RolloutBuffer.add_delayed_reward` `reward * discount**delay` uyguluyor mu?
- `discount < 1` iken aynı `event_turn`’de eski vs yeni launch’tan yeninin daha büyük kredi alması.
- Neutral capture base ödülü ship-share ile bölünüyor mu?
- Enemy capture `opponent_count_multiplier` uyguluyor mu?
- **Capture ödülü `production × remaining_steps / max_turns` formülünü kullanıyor mu** (normalize ölçek doğrulaması).
- **Planet loss penalty’si normalize edilmiş şekilde negatif değer üretiyor mu**.
- `production_simple` preset’i `fleet_airborne_weight = 0.0005` ve `delayed_reward_discount = 1.0` ile geliyor mu.
- GAE iki oyuncunun transition’larını ayrıştırıyor mu?

`tests/test_obs_builder.py`: planet feature sayısı 31, incoming bucket ve net_threat alanları beklenen indekslerde.

`tests/test_model.py`: tokenizer ve forward shape’leri; `tokens` çıkışının planet-only için `(B, P+1, d)` olduğu vs.

Bu testler her değişiklikte koşmalı (`pytest tests/`) çünkü PPO boruhattının çok fazla sessiz bozulabilir noktası vardır (feature index kaymaları, mask hataları, reward delay yanlış eklenmesi gibi).

## 10. Kısa Özet

- **Observation**: permutation-invariant set; geometrik edge bias; incoming fleet özetleri planet feature içinde.
- **Model**: biased Set Transformer; faktorize + maskeli aksiyon (src → tgt → ship_bin).
- **Öğrenme**: clipped PPO + GAE, kendi kendine oyun.
- **Ödül**: terminal + minimum step shaping + delayed credit. `production_simple`’da capture/loss `(T_max-t)/T_max` ile normalize, combat ships sayıları 100’e bölünmüş, airborne 0.0005, delayed discount 1.0.
- **Pratik**: default olarak `planet_only` ve `production_simple` preset; 1000 iterasyonluk GPU run ana referans, 10 iterasyonluk CPU smoke test hızlı doğrulama için. İlk birkaç epoch’ta `valL` ≪ 1000 ve `H` > 1 görmek gerekir; aksi halde reward kalibrasyonu kontrol edilmeli.

Daha derin okuma için: `docs/reward_system.md`, `docs/reward.md`, `docs/production_simple_reward.md`, `docs/math.md`.